# Curriculum, Self-Paced Learning, and Dataset Distillation (MTT)

### Historical Lineage & Theoretical Foundations
- **Curriculum Learning** (*Bengio et al., 2009 ICML*): Established that presenting training examples in increasing order of difficulty guides optimization into cleaner, more generalizable basins of attraction.
- **Self-Paced Learning (SPL)** (*Kumar, Packer & Koller, 2010 NeurIPS*): Formulates sample selection as a biconvex regularized optimization problem $\min_{w, v} \sum v_i \ell_i + g(\lambda, v_i)$, deriving analytical hard and linear pace-controlled sample weights.
- **Competence-Based Curricula** (*Platanios et al., 2019 NAACL*): Defines monotonic competence schedules $c(t)$ over training iterations, dynamically gating sample difficulty thresholds.
- **Dataset Distillation & Trajectory Matching** (*Wang et al., 2018; Zhao et al., 2021; Cazenavette et al., 2022 MTT*): Synthesizes tiny synthetic datasets ($1\text{--}10$ images per class) by matching student model gradients or unrolled training trajectories to expert models trained on real data.

> [!WARNING]
> **Decision Boundary (When NOT to Apply Online Hard Example Mining - OHEM):** Never use aggressive hard-example mining (OHEM) from epoch 0 on noisy datasets ($\ge 10\%$ label corruption). Mislabeled instances produce the highest prediction errors; OHEM exclusively samples corrupted noise, causing rapid generalization collapse. Always use Self-Paced Learning (SPL) or competence schedules to filter high-loss outliers early in training.

**Topic Connections:** `🔗 Upstream`: [Meta-Parameter Sensitivity & Effective LR](08_meta_parameter_sensitivity_fanova_sobol_meta_landscapes_and_effective_lr.ipynb) | `🎉 Complete`: Category 04 Training Methods & Optimisers Finished!


## 2. Mathematical Foundations & Governing Equations

1. **Self-Paced Learning Gating (Kumar et al., 2010):**
$$v_i^\text{hard} = \mathbb{I}(\ell_i < \lambda), \quad v_i^\text{linear} = \max\left(0, 1 - \frac{\ell_i}{\lambda}\right), \quad \mathcal{L}_\text{SPL} = \frac{\sum v_i \ell_i}{\sum v_i + \epsilon}$$
2. **Competence-Based Curriculum (Platanios et al., 2019):**
$$c(t) = \min\left(1.0, c_0 + (1 - c_0) \frac{t}{T_\text{warmup}}\right), \quad \text{mask}_i = \mathbb{I}(d_i \le c(t))$$
3. **Dataset Distillation via Gradient Matching (Zhao et al., 2021):**
$$\mathcal{L}_\text{GM} = 1 - \frac{\langle g_\text{synth}, g_\text{real} \rangle}{\|g_\text{synth}\|_2 \|g_\text{real}\|_2 + \epsilon}$$
4. **Matching Training Trajectories (MTT - Cazenavette et al., 2022):**
$$\mathcal{L}_\text{MTT} = \frac{\|\theta_{t+M}(\mathcal{S}) - \theta^*_{t+N}\|_2^2}{\|\theta^*_t - \theta^*_{t+N}\|_2^2 + \epsilon}$$
5. **Asymptotic Complexity:** SPL Gating $\mathcal{O}(N)$; Competence $\mathcal{O}(N)$; Gradient Matching $\mathcal{O}(D)$; Trajectory Matching $\mathcal{O}(M \cdot D)$.


In [ ]:
import sys
from pathlib import Path
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/04_training_methods_and_optimisers/03_training_dynamics_self_optimising_non_backprop/09_curriculum_self_paced_learning_and_dataset_distillation_mtt.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")


In [ ]:
class LinearAnchor(nn.Module):
    def __init__(self, in_features: int = 8, out_features: int = 4):
        super().__init__()
        self.fc = nn.Linear(in_features, out_features)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

def compute_spl_weights_and_loss(losses: torch.Tensor, pace: torch.Tensor) -> tuple[torch.Tensor, ...]:
    hw = (losses < pace).float()
    lw = torch.clamp(1.0 - losses / pace, min=0.0)
    lh = torch.sum(hw * losses) / (torch.sum(hw) + 1e-8)
    ll = torch.sum(lw * losses) / (torch.sum(lw) + 1e-8)
    return hw, lw, lh, ll

def compute_curriculum(diff: torch.Tensor, t: torch.Tensor, t_w: torch.Tensor, c0: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    comp = torch.clamp(c0 + (1.0 - c0) * (t / t_w), min=0.0, max=1.0)
    return comp, (diff <= comp).float()

def compute_grad_matching(gs: torch.Tensor, gr: torch.Tensor) -> torch.Tensor:
    cos = torch.sum(gs * gr) / (torch.sqrt(torch.sum(gs ** 2)) * torch.sqrt(torch.sum(gr ** 2)) + 1e-8)
    return 1.0 - cos

def compute_mtt_loss(t_start: torch.Tensor, t_stud: torch.Tensor, t_exp: torch.Tensor) -> torch.Tensor:
    return torch.sum((t_stud - t_exp) ** 2) / (torch.sum((t_start - t_exp) ** 2) + 1e-8)
print("Curriculum, SPL, and Dataset Distillation routines defined.")


In [ ]:
anchor = LinearAnchor(8, 4).to(device)
anchor.fc.weight.data.copy_(weights['anchor_w'].T)
anchor.fc.bias.data.copy_(weights['anchor_b'])

hw, lw, lh, ll = compute_spl_weights_and_loss(weights['losses'], weights['pace_lambda'])
err_spl = max(float(torch.max(torch.abs(hw - weights['golden_spl_hard_weights'])).item()),
              float(torch.max(torch.abs(lw - weights['golden_spl_linear_weights'])).item()),
              float(torch.abs(lh - weights['golden_weighted_loss_hard']).item()),
              float(torch.abs(ll - weights['golden_weighted_loss_linear']).item()))

comp, mask = compute_curriculum(weights['difficulties'], weights['step_t'], weights['t_warmup'], weights['c0'])
err_curriculum = max(float(torch.abs(comp - weights['golden_competence']).item()),
                     float(torch.max(torch.abs(mask - weights['golden_active_curriculum_mask'])).item()))

gm_loss = compute_grad_matching(weights['grad_synth'], weights['grad_real'])
err_gm = float(torch.abs(gm_loss - weights['golden_grad_matching_loss']).item())

mtt_loss = compute_mtt_loss(weights['theta_start'], weights['theta_student'], weights['theta_expert'])
err_mtt = float(torch.abs(mtt_loss - weights['golden_mtt_loss']).item())

anchor_out = anchor(weights['anchor_x'])
err_anchor = float(torch.max(torch.abs(anchor_out - weights['golden_anchor_out'])).item())

for name, err in [('SPL', err_spl), ('Curriculum', err_curriculum), ('GradMatching', err_gm), ('MTT', err_mtt), ('Anchor', err_anchor)]:
    assert err < 1e-5, f'{name} invariant failed: {err:.2e}'
print(f'All 5 invariants verified! SPL: {err_spl:.2e}, Curric: {err_curriculum:.2e}, GM: {err_gm:.2e}, MTT: {err_mtt:.2e}, Anchor: {err_anchor:.2e}')


In [ ]:
track_fns = {
    'Linear Anchor': lambda: anchor(weights['anchor_x']),
    'Self-Paced Gating': lambda: compute_spl_weights_and_loss(weights['losses'], weights['pace_lambda']),
    'Curriculum Mask': lambda: compute_curriculum(weights['difficulties'], weights['step_t'], weights['t_warmup'], weights['c0']),
    'Gradient Matching': lambda: compute_grad_matching(weights['grad_synth'], weights['grad_real']),
    'Trajectory Matching': lambda: compute_mtt_loss(weights['theta_start'], weights['theta_student'], weights['theta_expert']),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# Pathological Anti-Pattern: Online Hard Example Mining (OHEM) on data with 25% label corruption
# Clean sample losses ~ 0.2 - 0.8; Corrupted sample losses ~ 3.5 - 5.0
batch_losses = torch.tensor([0.25, 4.80, 0.40, 3.90, 0.30, 0.75, 4.20, 0.50])
is_corrupted = torch.tensor([0, 1, 0, 1, 0, 0, 1, 0]) # Samples 1, 3, 6 are corrupted

# OHEM selects top-3 highest loss samples:
_, ohem_indices = torch.topk(batch_losses, k=3)
ohem_corrupted_ratio = float(torch.sum(is_corrupted[ohem_indices]).item()) / 3.0 # 100% corrupted!

# Self-Paced Learning (SPL) with pace lambda = 1.0 filters out all high-loss outliers:
spl_mask = (batch_losses < 1.0).float()
spl_corrupted_ratio = float(torch.sum(is_corrupted * spl_mask).item()) / (float(torch.sum(spl_mask).item()) + 1e-8) # 0% corrupted!

print(f'[FAILURE DEMONSTRATED] OHEM top-k selection yielded: {ohem_corrupted_ratio * 100:.1f}% corrupted outliers!')
print(f'[CORRECT] Self-Paced Learning gating yielded: {spl_corrupted_ratio * 100:.1f}% corrupted outliers!')
assert ohem_corrupted_ratio == 1.0 and spl_corrupted_ratio == 0.0, 'Curriculum noise resilience test failed.'
print('[REMEDY] Use Self-Paced Learning rather than OHEM when training on noisy or web-crawled datasets.')


## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

- **Takeaways**: Curriculum and Self-Paced Learning protect optimization against noisy outliers; Dataset Distillation compresses thousands of instances into synthetic prototypes via trajectory alignment.

| Scientific Domain | Self-Paced Learning (SPL) | Competence Curricula | Trajectory Distillation (MTT) |
| :--- | :--- | :--- | :--- |
| **Large-Scale Pre-Training** | Filters corrupted web scrape tokens | Ramps up context length & perplexity | Distills knowledge bases into synthetic prompt buffers |
| **Robotics & Policy Learning** | Prunes non-reproducible simulator runs | Anneals domain randomization difficulty | Condenses expert demonstration logs into minimal trajectories |
| **Biomedical Drug Discovery** | Discards inconsistent bioassay noise | Stages simple to complex molecular scaffolds | Distills multi-million conformer libraries into prototype assays |

---
**Navigation:** [← Meta-Parameter Sensitivity & Effective LR](08_meta_parameter_sensitivity_fanova_sobol_meta_landscapes_and_effective_lr.ipynb) | [↑ Master Index](../../../README.md#04_training_methods_and_optimisers) | [Category 04 Complete! 🎉](../../../README.md#04_training_methods_and_optimisers)
